# 07 · Forced alignment

MMS CTC on romanized text (uroman) → word timestamps for Arabic **and** English words. Clips are kept only if the alignment score is good, every word is timed and no word touches the clip edge (no word cut in the middle).

In [ ]:
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)
if Path("/workspace").exists():
    os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
import numpy as np, pandas as pd
import plotly.express as px
from egyspeech import nb
cfg, lay = nb.cfg_and_layout()          # configs/config.yaml
print("work dir:", lay.root)

In [ ]:
nb.run("align")

In [ ]:
from egyspeech.io import iter_jsonl_dir
al = pd.DataFrame(list(iter_jsonl_dir(lay.meta / "aligned")))
print(len(al), "clips |", f"{al.align_ok.mean():.1%} aligned OK | edge OK {al.edge_ok.mean():.1%}")
fig = px.histogram(al, x="align_score", nbins=60, title="Alignment score"); fig.add_vline(x=cfg.alignment.min_score, line_color="red"); fig.show()

In [ ]:
clips = {r["id"]: r for r in nb.read_jsonl(lay.filtered)}
r = al[al.align_ok].iloc[0]
from egyspeech.io import read_audio
wav, sr = read_audio(clips[r.id]["path"])
t = np.arange(len(wav)) / sr
fig = px.line(x=t[::50], y=wav[::50], title=f"{r.id}: word boundaries")
for w in r.words:
    if w["start"] is not None:
        fig.add_vrect(x0=w["start"], x1=w["end"], opacity=0.15, line_width=0, annotation_text=w["word"], annotation_position="top left")
fig.show(); nb.play(clips[r.id]["path"])